Base Imports & State (Prerequisites)

In [1]:
import requests
from typing import Optional, Dict, Any, TypedDict
from pydantic import BaseModel, Field

# --- STATE SCHEMA ---
class WeatherAgentState(TypedDict):
    name: str
    location_data: Optional[Dict[str, Any]]
    weather_data: Optional[Dict[str, Any]]
    weather_info: Optional[str]

Debugging Fix 1 - The Configuration

In [2]:

from pydantic_settings import BaseSettings
from typing import ClassVar, Dict

class Config(BaseSettings):
    LOCATION_API_URL: str = "https://ipapi.co/json/"
    WEATHER_API_BASE_URL: str = "https://api.open-meteo.com/v1/forecast"
    REQUEST_TIMEOUT: int = 10
    MAX_RETRIES: int = 3
    
    TEMP_COLD: float = 10.0
    TEMP_COOL: float = 18.0
    TEMP_COMFORTABLE: float = 26.0
    TEMP_WARM: float = 32.0
    
    WEATHER_CODE_DESCRIPTIONS: ClassVar[Dict[int, str]] = {
        0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
        # ... (truncated for brevity, but all codes are loaded)
        95: "Thunderstorm"
    }

    # The corrected syntax that prevents crashes
    TEMP_MIN: str = "Needs Debugging"
    
    model_config = {
        "env_file": ".env",
        "case_sensitive": False,
        "extra": "ignore"
    }

config = Config()

# Debug Output to verify Fix 1:
print(f"Weather API URL: {config.WEATHER_API_BASE_URL}")

Weather API URL: https://api.open-meteo.com/v1/forecast


Debugging Fix 2 - Helper Functions

In [3]:
# --- FIX 2: HELPER LOGIC ---
from datetime import datetime, timezone, timedelta

def classify_temperature(temp_celsius: float) -> str:
    # Fixed: Removed the faulty `if temp_celsius:` override
    if temp_celsius < config.TEMP_COLD: return "cold"
    elif temp_celsius < config.TEMP_COOL: return "cool"
    elif temp_celsius < config.TEMP_COMFORTABLE: return "comfortable"
    elif temp_celsius < config.TEMP_WARM: return "warm"
    else: return "hot"

def get_greeting(is_day: int) -> str:
    # Fixed: Safely cast to boolean instead of strict integer check
    return "Good morning" if bool(is_day) else "Good evening"

# (Include parse_utc_offset and format_local_time here as well)

# Debug Output to verify Fix 2:
print(f"20°C is considered: {classify_temperature(20.0)}")
print(f"Greeting for night (0): {get_greeting(0)}")

20°C is considered: comfortable
Greeting for night (0): Good evening


Debugging Fix 3 - Location Node & Fallbacks

In [4]:

def _normalize_location_payload(payload: dict, provider: str) -> dict:
    if provider == "ipapi":
        return {"city": payload["city"], "region": payload["region"], "country_name": payload["country_name"], "latitude": payload["latitude"], "longitude": payload["longitude"], "utc_offset": payload["utc_offset"], "timezone": payload["timezone"]}
    if provider == "ipwhois":
        timezone_data = payload.get("timezone", {})
        return {"city": payload["city"], "region": payload["region"], "country_name": payload["country"], "latitude": payload["latitude"], "longitude": payload["longitude"], "utc_offset": timezone_data.get("utc", "+00:00"), "timezone": timezone_data.get("id", "UTC")}

def _fetch_location_from_url(url: str) -> dict:
    response = requests.get(url, timeout=config.REQUEST_TIMEOUT)
    response.raise_for_status()
    payload = response.json()
    if "ipwho.is" in url:
        return _normalize_location_payload(payload, "ipwhois")
    return _normalize_location_payload(payload, "ipapi")

def fetch_location_data(state: WeatherAgentState) -> WeatherAgentState:
    last_error = None
    for url in (config.LOCATION_API_URL, "https://ipwho.is/"):
        try:
            state["location_data"] = _fetch_location_from_url(url)
            return state
        except Exception as e:
            last_error = e
    raise last_error or Exception("Failed to resolve location")

# Debug Output to verify Fix 3:
test_state = WeatherAgentState(name="Debug", location_data=None, weather_data=None, weather_info=None)
test_state = fetch_location_data(test_state)
print("Location Fetched:", test_state["location_data"]["city"])

Location Fetched: Mumbai


Debugging Fix 4 - String Formatting & Weather Node

In [5]:

import requests

def fetch_weather_data(state: WeatherAgentState) -> WeatherAgentState:
    """Fetch real weather data to populate the state."""
    if not state.get("location_data"):
        raise Exception("Location data not available for weather fetch")

    location = state["location_data"]

    try:
        params = {
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current_weather": "true",
        }

        response = requests.get(
            config.WEATHER_API_BASE_URL,
            params=params,
            timeout=config.REQUEST_TIMEOUT,
        )
        response.raise_for_status()

        weather_data = response.json()
        state["weather_data"] = weather_data # This fixes the NoneType error!

    except Exception as e:
        raise Exception(f"Unexpected error fetching weather: {str(e)}")

    return state

def generate_weather_info(state: WeatherAgentState) -> WeatherAgentState:
    """Format the retrieved data into our fixed string output."""
    location = state["location_data"]
    weather = state["weather_data"]["current_weather"]
    units = state["weather_data"].get("current_weather_units", {})

    temperature = weather["temperature"]
    temp_unit = units.get("temperature", "C")
    windspeed = weather["windspeed"]
    
    # Fixed: Unit assignment and corrected f-string
    wind_unit = units.get("windspeed", "km/h")
    temp_classification = classify_temperature(temperature)

    weather_info_parts = [
        f"Your current location: {location['city']}",
        f"- Temperature: {temperature}{temp_unit} ({temp_classification})",
        f"- Wind: {windspeed} {wind_unit}", 
    ]

    state["weather_info"] = "\n".join(weather_info_parts)
    return state

# --- Debug Output to verify Fix 4 ---
# (Assuming test_state still has the location data from Cell 4)
print("Fetching live weather data...")
test_state = fetch_weather_data(test_state)

print("Generating weather info...")
test_state = generate_weather_info(test_state)

print("\nSuccess! Here is the formatted output:")
print("-" * 40)
print(test_state["weather_info"])
print("-" * 40)


Fetching live weather data...
Generating weather info...

Success! Here is the formatted output:
----------------------------------------
Your current location: Mumbai
- Temperature: 33.0°C (hot)
- Wind: 15.5 km/h
----------------------------------------


Debugging Fix 5 - Graph Execution

In [6]:

from langgraph.graph import StateGraph, START, END

# 1. Build the graph
builder = StateGraph(WeatherAgentState)
builder.add_node("fetch_location_data", fetch_location_data)
builder.add_node("fetch_weather_data", fetch_weather_data)
builder.add_node("generate_weather_info", generate_weather_info)

builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "fetch_weather_data") # Note: Ensure fetch_weather_data is chained here
builder.add_edge("fetch_weather_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)

weather_agent = builder.compile()

# 2. Execute interactively
final_state = weather_agent.invoke({
    "name": "Jupyter User",
    "location_data": None,
    "weather_data": None,
    "weather_info": None
})

print("\n" + "=" * 40)
print("FINAL GRAPH OUTPUT")
print("=" * 40)
print(final_state["weather_info"])


FINAL GRAPH OUTPUT
Your current location: Delhi
- Temperature: 35.4°C (hot)
- Wind: 6.3 km/h
